# 02 — Training (EfficientNet-B0 and ResNet-18)

Run on Colab with **Runtime → Change runtime type → GPU**.

Theory: [`docs/03_model_architecture.md`](../docs/03_model_architecture.md), [`docs/04_training.md`](../docs/04_training.md)

In [ ]:
# --- environment setup: works on Kaggle, Colab and locally ---
import sys, os, glob
REPO = "https://github.com/Mohd2Shaan/Deepfake-Detection-Project.git"
if os.path.exists("/kaggle/input"):                      # ---- Kaggle
    if not os.path.exists("/kaggle/working/project"):
        !git clone -q {REPO} /kaggle/working/project
    %cd /kaggle/working/project
    !pip install -q grad-cam
    # use the dataset attached via "Add Input" directly (read-only, nothing to download)
    for depth in range(1, 5):
        hits = [os.path.dirname(t) for t in glob.glob("/kaggle/input/" + "*/" * depth + "train")
                if os.path.isdir(os.path.join(os.path.dirname(t), "valid"))]
        if hits:
            os.environ["DEEPFAKE_DATA_DIR"] = hits[0]
            break
    else:
        print("WARNING: dataset not found - attach '140k Real and Fake Faces' via Add Input")
elif "google.colab" in sys.modules:                       # ---- Colab
    if not os.path.exists("/content/project"):
        !git clone -q {REPO} /content/project
    %cd /content/project
    !pip install -q -r requirements.txt kagglehub
elif os.path.basename(os.getcwd()) == "notebooks":        # ---- local
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

In [ ]:
# --- dataset (Colab/local: downloads ~4 GB via kagglehub the first time; Kaggle: already attached) ---
from src import config
if not (config.DATA_DIR / "train").exists():
    !python scripts/download_dataset.py --move
print("data dir:", config.DATA_DIR)

In [ ]:
# --- checkpoints: Colab -> Google Drive (survives disconnects); Kaggle/local -> checkpoints/ ---
import sys
CKPT_DIR = "checkpoints"
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_DIR = "/content/drive/MyDrive/deepfake-detection/checkpoints"
    !mkdir -p {CKPT_DIR}
print("checkpoints ->", CKPT_DIR)

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## Model A — EfficientNet-B0
3 frozen epochs (head only, lr 1e-4) → 7 fine-tune epochs (last 2 blocks, lr 1e-5).

If Colab disconnects, re-run the setup cells and add `--resume`.

In [ ]:
!python -m src.train --model efficientnet_b0 --checkpoint-dir {CKPT_DIR} --num-workers 4

## Model B — ResNet-18

In [ ]:
!python -m src.train --model resnet18 --checkpoint-dir {CKPT_DIR} --num-workers 4

## Training curves

In [ ]:
from IPython.display import Image as Show, display
for name in ['efficientnet', 'resnet']:
    display(Show(f'outputs/plots/training_curves_{name}.png'))

In [ ]:
# copy best checkpoints back into the project folder for evaluation / the app
!mkdir -p checkpoints && cp {CKPT_DIR}/*_best.pth checkpoints/ 2>/dev/null; ls -lh checkpoints

## Observations

_Note convergence speed, gap between train/valid, effect of unfreezing at epoch 4._